In [ ]:
import json
import pandas as pd

df = pd.read_json('data/oracle-cards.jsonl', lines=True)

In [ ]:
df.columns

In [ ]:
df = df[['oracle_id', 'name', 'colors']]
df['colors'] = df['colors'].apply(lambda x: x if isinstance(x, list) else [])
df['colors'] = df['colors'].apply(lambda x: 'M' if len(x) > 1 else x[0] if x else 'C')
df.head()

In [ ]:
tag_df = pd.read_json('data/oracle-tags.jsonl', lines=True)

In [ ]:
tag_df.columns

In [ ]:
tag_df = tag_df[['label', 'taggings']]
tag_dict = {}
tag_labels = []
for index, row in tag_df.iterrows():
    taggings = row['taggings']
    label = row['label']
    tag_labels.append(label)
    if isinstance(taggings, list):
        for i in range(len(taggings)):
            if(taggings[i]['oracle_id'] not in tag_dict):
                tag_dict[taggings[i]['oracle_id']] = []
            tag_dict[taggings[i]['oracle_id']].append(label)

In [ ]:
tagging_row_indices = []
tagging_col_indices = []
id_list = df['oracle_id'].tolist()
for key, value in tag_dict.items():
    id_index = id_list.index(key)
    for label in value:
        tagging_row_indices.append(id_index)
        tagging_col_indices.append(tag_labels.index(label))

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

data = np.ones(len(tagging_row_indices), dtype=int)
sparse_mat = csr_matrix(
    (data, (tagging_row_indices, tagging_col_indices)),
    shape=(len(df), len(tag_labels)),
    dtype=int
)

df_sparse = pd.DataFrame.sparse.from_spmatrix(sparse_mat, index=df.index, columns=tag_labels)

df_with_tags = pd.concat([df, df_sparse], axis=1)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support


x_train, x_test, y_train, y_test = train_test_split(df_with_tags[tag_labels], df_with_tags['colors'], test_size=0.2, random_state=42)

classifier = MultinomialNB()
classifier.fit(x_train, y_train)

y_pred = classifier.predict(x_test)
p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='weighted')
print(p, r, f)